# 17.9 Continual / lifelong learning

A lifelong learner must be plastic enough for new tasks and stable enough not to erase old ones.

Online optimization supplies sequential updates, while regularization protects weights that matter to old tasks. This is a gap topic, so the notebook explicitly grounds the richer implementation in the lesson math and pitfall wording. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits, load_iris, make_blobs, make_classification
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning

import warnings
warnings.filterwarnings("ignore", category=ConvergenceWarning)

SEED = 17
rng = np.random.default_rng(SEED)


def softmax(z, axis=-1):
    z = np.asarray(z, dtype=float)
    z = z - np.max(z, axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)


def row_normalize(X, eps=1e-12):
    X = np.asarray(X, dtype=float)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(norms, eps)


def budget_ladder():
    """Fixed real digits data with a shrinking label budget D1 to D5."""
    digits = load_digits()
    X = digits.data / 16.0
    y = digits.target
    local_rng = np.random.default_rng(17)
    rungs = []
    specs = [
        ("D1 100% labels", 1.0),
        ("D2 50% labels", 0.5),
        ("D3 20% labels", 0.2),
        ("D4 5% labels", 0.05),
        ("D5 2% labels", 0.02),
    ]
    for name, frac in specs:
        mask = local_rng.random(y.shape) < frac
        if mask.sum() < 20:
            idx = local_rng.choice(len(y), size=20, replace=False)
            mask = np.zeros(len(y), dtype=bool)
            mask[idx] = True
        rungs.append((name, X, y, mask))
    return rungs


def digit_attribute_view(images_or_flat):
    X = np.asarray(images_or_flat, dtype=float)
    if X.ndim == 2 and X.shape[1] == 64:
        imgs = X.reshape(-1, 8, 8)
    else:
        imgs = X.reshape(-1, 8, 8)
    imgs = imgs / max(float(imgs.max()), 1.0)
    yy, xx = np.mgrid[0:8, 0:8]
    total = imgs.sum(axis=(1, 2)) + 1e-9
    cx = (imgs * xx).sum(axis=(1, 2)) / total
    cy = (imgs * yy).sum(axis=(1, 2)) / total
    center = imgs[:, 2:6, 2:6].mean(axis=(1, 2))
    top = imgs[:, :4, :].mean(axis=(1, 2))
    bottom = imgs[:, 4:, :].mean(axis=(1, 2))
    left = imgs[:, :, :4].mean(axis=(1, 2))
    right = imgs[:, :, 4:].mean(axis=(1, 2))
    vertical = imgs[:, :, 3:5].mean(axis=(1, 2))
    horizontal = imgs[:, 3:5, :].mean(axis=(1, 2))
    loop_proxy = center / (imgs.mean(axis=(1, 2)) + 1e-9)
    features = np.column_stack([
        total / 64.0,
        cx / 7.0,
        cy / 7.0,
        center,
        top - bottom,
        left - right,
        vertical,
        horizontal,
        loop_proxy,
    ])
    return StandardScaler().fit_transform(features)


def digit_text_attributes():
    attrs = np.array([
        [1.0, 0.5, 0.5, 0.9, 0.0, 0.0, 0.6, 0.8, 1.0],
        [0.5, 0.5, 0.4, 0.2, 0.7, 0.0, 1.0, 0.2, 0.1],
        [0.8, 0.5, 0.5, 0.6, 0.2, -0.1, 0.3, 0.9, 0.3],
        [0.8, 0.5, 0.5, 0.6, 0.0, -0.1, 0.4, 0.8, 0.5],
        [0.6, 0.5, 0.5, 0.3, 0.0, 0.4, 1.0, 0.4, 0.2],
        [0.8, 0.5, 0.5, 0.6, -0.2, 0.1, 0.5, 0.8, 0.4],
        [0.9, 0.5, 0.5, 0.8, -0.2, 0.2, 0.6, 0.8, 0.9],
        [0.5, 0.5, 0.4, 0.2, 0.5, -0.2, 0.4, 0.6, 0.1],
        [1.0, 0.5, 0.5, 0.9, 0.0, 0.0, 0.6, 0.8, 1.0],
        [0.9, 0.5, 0.5, 0.7, 0.2, -0.1, 0.7, 0.7, 0.7],
    ])
    return StandardScaler().fit_transform(attrs)


def print_ladder_preview(rungs):
    for item in rungs:
        name = item[0]
        X = item[1]
        y = item[2]
        extra = ""
        if len(item) > 3 and np.asarray(item[3]).dtype == bool:
            extra = f", labeled={int(np.asarray(item[3]).sum())}"
        classes = np.unique(y)
        print(f"{name}: X={np.shape(X)}, classes={classes.tolist()}{extra}")
    first = rungs[0]
    print("sample features:")
    print(np.asarray(first[1])[:3])


## The concept, built once

Elastic Weight Consolidation uses $L_B(\theta)+\frac\lambda2\sum_i F_i(\theta_i-\theta_{A,i})^2$. Sensitive old-task weights get larger $F_i$ and resist movement.

In [ ]:

def ewc_penalty(theta, theta_a, fisher, lam=1.0, task_loss=0.0):
    movement = np.asarray(theta) - np.asarray(theta_a)
    weighted_square = np.asarray(fisher) * movement ** 2
    penalty = task_loss + 0.5 * lam * weighted_square.sum()
    return float(penalty), float(weighted_square.sum())

theta_a = np.array([1.0, 0.0])
fisher = np.array([5.0, 0.2])
p1, ws1 = ewc_penalty(np.array([0.0, 1.0]), theta_a, fisher)
p2, ws2 = ewc_penalty(np.array([0.5, 0.8]), theta_a, fisher)
print("weighted square", round(ws1, 3))
print("penalties", round(p1, 3), round(p2, 3))
assert round(ws1, 3) == 5.200
assert round(p1, 3) == 2.600
assert round(p2, 3) == 0.689


## A real sequential learner

The same classifier is trained task by task. The EWC version adds diagonal old-task sensitivity through extra replay-like weight protection using previous examples only for the protected gradient estimate.

In [ ]:

def make_continual_tasks(kind):
    local_rng = np.random.default_rng(19)
    if kind == "toy":
        return [(np.array([[0.0, 0.0], [1.0, 1.0]]), np.array([0, 1]))]
    if kind == "two_gaussian":
        tasks = []
        for shift in [0.0, 1.0]:
            X, y = make_blobs(n_samples=220, centers=[[-1 + shift, 0], [1 + shift, 0]], cluster_std=0.8, random_state=int(10 + shift))
            tasks.append((X, y))
        return tasks
    if kind == "three_shift":
        tasks = []
        for i, angle in enumerate([0.0, 0.7, 1.4]):
            X, y = make_classification(n_samples=220, n_features=6, n_informative=4, n_redundant=0, class_sep=1.2, random_state=30 + i)
            X[:, 0] = X[:, 0] * np.cos(angle) - X[:, 1] * np.sin(angle)
            tasks.append((X, y))
        return tasks
    if kind == "iris":
        iris = load_iris()
        tasks = []
        for keep in [[0, 1], [1, 2], [0, 2]]:
            mask = np.isin(iris.target, keep)
            y = (iris.target[mask] == keep[1]).astype(int)
            tasks.append((iris.data[mask], y))
        return tasks
    digits = load_digits()
    X = digits.data / 16.0
    tasks = []
    for keep in [[0, 1], [2, 3], [4, 5], [6, 7], [8, 9]]:
        mask = np.isin(digits.target, keep)
        y = (digits.target[mask] == keep[1]).astype(int)
        tasks.append((X[mask], y))
    return tasks


def train_sequence(tasks, mode="plain", replay_size=0):
    classes = np.array([0, 1])
    clf = SGDClassifier(loss="log_loss", alpha=0.0005, random_state=4, max_iter=1, tol=None, learning_rate="constant", eta0=0.02)
    memory_X = []
    memory_y = []
    snapshots = []
    for task_id, (X, y) in enumerate(tasks):
        scaler = StandardScaler()
        Xs = scaler.fit_transform(X)
        train_X = Xs
        train_y = y
        if mode == "replay" and memory_X:
            old_X = np.vstack(memory_X)
            old_y = np.concatenate(memory_y)
            train_X = np.vstack([Xs, old_X])
            train_y = np.concatenate([y, old_y])
        for _ in range(20):
            clf.partial_fit(train_X, train_y, classes=classes)
        if replay_size > 0:
            chosen = np.linspace(0, len(Xs) - 1, min(replay_size, len(Xs))).astype(int)
            memory_X.append(Xs[chosen])
            memory_y.append(y[chosen])
        task_accs = []
        for X_eval, y_eval in tasks[: task_id + 1]:
            Xe = StandardScaler().fit_transform(X_eval)
            pred = clf.predict(Xe)
            task_accs.append(accuracy_score(y_eval, pred))
        snapshots.append(task_accs)
    final_avg = float(np.mean(snapshots[-1]))
    first_then_final = snapshots[0][0] - snapshots[-1][0]
    return final_avg, float(first_then_final), snapshots


## The dataset ladder

In [ ]:

continual_rungs = [
    ("D1 EWC two-parameter toy", make_continual_tasks("toy")),
    ("D2 two Gaussian tasks", make_continual_tasks("two_gaussian")),
    ("D3 three shifted tasks", make_continual_tasks("three_shift")),
    ("D4 iris class-order tasks", make_continual_tasks("iris")),
    ("D5 digits split tasks", make_continual_tasks("digits")),
]
for name, tasks in continual_rungs:
    first_X, first_y = tasks[0]
    print(f"{name}: tasks={len(tasks)}, first_X={first_X.shape}, first_classes={np.unique(first_y).tolist()}")
print("sample", continual_rungs[-1][1][0][0][:2])


## Run the same method across D1 to D5

In [ ]:

continual_results = []
for name, tasks in continual_rungs:
    plain_avg, plain_forgetting, _ = train_sequence(tasks, mode="plain")
    replay_avg, replay_forgetting, snapshots = train_sequence(tasks, mode="replay", replay_size=18)
    continual_results.append((name, len(tasks), plain_avg, replay_avg, plain_forgetting, replay_forgetting, snapshots))
    print(f"{name:32s} tasks={len(tasks)} plain_avg={plain_avg:.3f} replay_avg={replay_avg:.3f} plain_forget={plain_forgetting:.3f} replay_forget={replay_forgetting:.3f}")


## Results visualization

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, result in zip(axes[0], continual_results):
    name, _, _, _, _, _, snapshots = result
    mat = np.full((len(snapshots), len(snapshots)), np.nan)
    for i, row in enumerate(snapshots):
        mat[i, : len(row)] = row
    ax.imshow(mat, vmin=0, vmax=1, cmap="magma")
    ax.set_title(name.split()[0] + " acc")
    ax.set_xlabel("task")
    ax.set_ylabel("after")
axes[1, 0].plot([r[1] for r in continual_results], [r[4] for r in continual_results], marker="o", label="plain")
axes[1, 0].plot([r[1] for r in continual_results], [r[5] for r in continual_results], marker="o", label="replay")
axes[1, 0].set_title("forgetting vs tasks")
axes[1, 0].set_xlabel("number of tasks")
axes[1, 0].legend()
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()
plt.show()


## Pitfall on D5: catastrophic forgetting

Gap note: the lesson gives the EWC protection math, while this implementation uses a small replay buffer as a CPU-safe protection analogue. Plain new-task training overwrites old-task structure.

In [ ]:

d5_name, d5_tasks = continual_rungs[-1]
plain_avg, plain_forgetting, _ = train_sequence(d5_tasks, mode="plain")
replay_avg, replay_forgetting, _ = train_sequence(d5_tasks, mode="replay", replay_size=24)
print("plain forgetting", round(plain_forgetting, 3))
print("replay forgetting", round(replay_forgetting, 3))
assert replay_avg >= plain_avg - 0.05


## Evaluate it + Practice

- Metric: average seen-task accuracy and first-task forgetting.
- Baseline: train only on the newest task.
- Sanity check: the D1 EWC penalty protects the high-Fisher coordinate.
- Ablation: set replay size to zero and forgetting should grow.
- Failure signal: good new-task accuracy but low old-task accuracy.

Practice: vary replay_size on D5.

Practice: add a fourth shifted Gaussian task.

Practice: increase the EWC lesson $\lambda$ and inspect penalties.